In [ ]:
# imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score, roc_auc_score

# Configuración
SEMILLA = 100522258
ARCHIVO_DATOS = '../data/bank_13.pkl'

In [ ]:
df = pd.read_pickle(ARCHIVO_DATOS)
print(f"Dimensiones: {df.shape}")
df.head()

In [ ]:
def simple_eda(data):
    print("--- Análisis de Variables ---")
    print(f"Instancias: {data.shape[0]}, Variables: {data.shape[1]}")
    
    # Tipos de variables
    cat_cols = data.select_dtypes(include=['object', 'category']).columns.tolist()
    num_cols = data.select_dtypes(include=['number']).columns.tolist()
    
    print(f"Numéricas: {num_cols}")
    print(f"Categóricas: {cat_cols}")
    
    # Cardinalidad y Faltantes
    summary = pd.DataFrame({
        'nulos': data.isnull().sum(),
        'unicos': data.nunique(),
        'tipo': data.dtypes
    })
    print("\nResumen de columnas:")
    print(summary)
    
    # Balanceo de target
    if 'deposit' in data.columns:
        print("\nBalanceo de la variable objetivo 'deposit':")
        print(data['deposit'].value_counts(normalize=True))

simple_eda(df)

In [ ]:
# pdays: -1 significa que nunca fue contactado. 
df['pdays_never'] = (df['pdays'] == -1).astype(int)

In [ ]:
X = df.drop(columns=['deposit'])
y = df['deposit']

# Holdout: 2/3 train, 1/3 test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=SEMILLA, stratify=y
)

print(f"Tamaño Train: {X_train.shape}, Tamaño Test: {X_test.shape}")

In [ ]:
numeric_features = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

# Preprocesador base (usaremos Standard por defecto, luego comparamos)
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

In [ ]:
# Probaremos cuál escalador funciona mejor para KNN
results_scaling = {}
for name, scaler in [('minmax', MinMaxScaler()), ('standard', StandardScaler()), ('robust', RobustScaler())]:
    pipe_knn = Pipeline([
        ('prep', ColumnTransformer([
            ('num', scaler, numeric_features),
            ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
        ])),
        ('knn', KNeighborsClassifier())
    ])
    # Evaluación interna (Inner) usando validación cruzada en Train
    from sklearn.model_selection import cross_val_score
    score = cross_val_score(pipe_knn, X_train, y_train, cv=5, scoring='f1_macro').mean()
    results_scaling[name] = score

print(f"Resultados de escalado para KNN: {results_scaling}")
# Elige el mejor para el pipeline definitivo